<a href="https://colab.research.google.com/github/swarnkarnitin/TrafficMonitoring/blob/main/OpevCV_DNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


To export yolo model as onnx

In [3]:
# from ultralytics import YOLO

# # Load your trained YOLOv8 model
# model = YOLO('/content/drive/MyDrive/Traffic_Videos/Trained_Model/best.pt')

# # Export the model to ONNX format
# # The exported model will be saved in the same directory as the original weights file
# model.export(format='onnx')

# print("Model exported to ONNX successfully!")

In [4]:
import cv2

# Path to your ONNX model
model_path = '/content/drive/MyDrive/Traffic_Videos/Trained_Model/best.onnx'

# Load the ONNX model
net = cv2.dnn.readNetFromONNX(model_path)

print("ONNX model loaded successfully!")

ONNX model loaded successfully!


In [14]:
import cv2

# Path to the input video
video_path = '/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part4.MOV'

# Open the video file
cap = cv2.VideoCapture(video_path)

# Get video properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

print(f"Video opened successfully: {video_path}")
print(f"Frame Width: {frame_width}")
print(f"Frame Height: {frame_height}")
print(f"FPS: {fps}")

Video opened successfully: /content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part4.MOV
Frame Width: 640
Frame Height: 480
FPS: 25


In [8]:
# Set the preferred backend and target
# You might need to change these based on your edge device
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_DEFAULT)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA) # Change to cv2.dnn.DNN_TARGET_CUDA for GPU

print("OpenCV DNN backend and target set.")

OpenCV DNN backend and target set.


In [15]:
# Define the output video path and codec
output_video_path = '/content/drive/MyDrive/Traffic_Videos/Mundhawan_Circle_SLR_1_Part4.avi' # Changed output file name
fourcc = cv2.VideoWriter_fourcc(*'MJPG') # Changed codec to MJPG
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# Get the names of the output layers
output_layer_names = net.getUnconnectedOutLayersNames()

# Initialize lists for class names and colors (replace with your actual class names)
classes = ['bus', 'car', 'microbus', 'motorbike', 'pickup-van', 'truck'] # Replace with your class names
colors = [(0, 255, 0), (0, 0, 255), (255, 0, 0)] # Add more colors if you have more classes

import numpy as np # Import numpy

frame_count = 0 # Initialize frame count for debugging

print(f"Video capture opened: {cap.isOpened()}") # Check if video capture is opened

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print(f"Failed to read frame {frame_count}. Exiting loop.") # Print if frame reading fails
        break

    frame_count += 1 # Increment frame count
    if frame_count % 100 == 0: # Print progress every 100 frames
        print(f"Processing frame {frame_count}...")


    # --- Preprocess the frame ---
    # Create a 4D blob from the frame
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (640, 640), swapRB=True, crop=False)

    # Set the input to the network
    net.setInput(blob)

    # --- Run inference ---
    # Run forward pass to get output of the output layers
    outs = net.forward(output_layer_names)

    # --- Postprocess the output ---
    # Initialize lists for detected bounding boxes, confidences, and class IDs
    boxes = []
    confidences = []
    class_ids = []

    # The output shape is (batch_size, num_features, num_detections)
    # We need to iterate through the detections (last dimension)
    # and extract bounding box and scores from the features (second dimension)
    output = outs[0].transpose(0, 2, 1) # Transpose to (batch_size, num_detections, num_features)
    output = np.squeeze(output, axis=0) # Remove batch_size dimension if batch_size is 1

    # Loop over each detection
    for detection in output:
        # The first 4 elements are bounding box coordinates (center_x, center_y, width, height)
        # The remaining elements are class scores
        scores = detection[4:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]

        # Filter out weak predictions
        if confidence > 0.5: # You can adjust the confidence threshold
            # Scale the bounding box coordinates back to the original image size
            center_x = int(detection[0] * frame_width)
            center_y = int(detection[1] * frame_height)
            width = int(detection[2] * frame_width)
            height = int(detection[3] * frame_height)
            left = int(center_x - width / 2)
            top = int(center_y - height / 2)

            boxes.append([left, top, width, height])
            confidences.append(float(confidence))
            class_ids.append(class_id)

    # Print number of detections before NMS for debugging
    # print(f"Frame {frame_count}: Number of detections before NMS: {len(boxes)}")


    # Apply Non-Maximum Suppression to remove redundant overlapping boxes
    indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4) # You can adjust the thresholds

    # Print number of detections after NMS for debugging
    # print(f"Frame {frame_count}: Number of detections after NMS: {len(indices) if indices is not None else 0}")

    # --- Visualize the results ---
    if indices is not None and len(indices) > 0: # Check if indices is not None and has elements
        for i in indices.flatten():
            box = boxes[i]
            left, top, width, height = box
            color = colors[class_ids[i] % len(colors)]
            cv2.rectangle(frame, (left, top), (left + width, top + height), color, 2)
            cv2.putText(frame, f"{classes[class_ids[i]]}: {confidences[i]:.2f}", (left, top - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Print detection details for debugging
            # print(f"  Detection: Class ID: {class_ids[i]}, Confidence: {confidences[i]:.2f}, Box: [{left}, {top}, {width}, {height}]")


    # Write the frame with detections to the output video
    out.write(frame)

# Release the video capture and writer objects
cap.release()
out.release()

print(f"Processing complete. Output video saved to: {output_video_path}")

Video capture opened: True
Processing frame 100...
Processing frame 200...
Processing frame 300...
Processing frame 400...
Processing frame 500...
Processing frame 600...
Processing frame 700...
Processing frame 800...
Processing frame 900...
Processing frame 1000...
Processing frame 1100...
Processing frame 1200...
Processing frame 1300...
Processing frame 1400...
Processing frame 1500...
Processing frame 1600...
Processing frame 1700...
Processing frame 1800...
Processing frame 1900...
Processing frame 2000...
Processing frame 2100...
Processing frame 2200...
Processing frame 2300...
Processing frame 2400...
Processing frame 2500...
Processing frame 2600...
Processing frame 2700...
Processing frame 2800...
Processing frame 2900...
Processing frame 3000...
Processing frame 3100...
Processing frame 3200...
Processing frame 3300...
Processing frame 3400...
Processing frame 3500...
Processing frame 3600...
Processing frame 3700...
Processing frame 3800...
Processing frame 3900...
Process